<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/04_phase3/MaxVit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – MaxVit Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions (text_qwen3-4b)
  - Hybrid captions (hybrid_gemma3-4b)
  - Vision-generated captions (vision_gemma3-4b)
  
The impact of different lightweight text encoders (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is MaxVit, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [1]:
!pip install -q timm transformers wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs("/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions", exist_ok=True)

In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines all experimental settings used throughout the notebook.
Hyperparameters include sample size, image size, batch size, number of epochs,
learning rate, weight decay, gradient clipping, and early stopping patience.

The DataLoader is configured via num_workers. The Dataset uses max_token_len
to control tokenization length. The regression head is configured via
regressor_hidden_dim and regressor_dropout. W&B project names and the
predictions output path are also centralized here to avoid hardcoding.

The target variables are the seven classes: Tree, Shrub, Grass, Crop,
Built-up, Barren, and Water. The text_scale parameter controls the strength of
the textual contribution during gated fusion. The random seed is set here and
applied globally for reproducibility.

In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "maxvit_tiny_tf_224",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7,

    # DataLoader
    "num_workers": 2,

    # Dataset
    "max_token_len": 128,

    # Model
    "regressor_hidden_dim": 256,
    "regressor_dropout": 0.25,

    # W&B
    "wandb_project": "DI725-Project_phase3_experiments",
    "wandb_project_scale": "DI725-Project_phase3_text_scale_exp",

    # Predictions output path
    "predictions_dir": "/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions",
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_475/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_475/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_475/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land-cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [11]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land cover classes remain consistent across splits, preventing bias in training or evaluation.

In [12]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [13]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [14]:
train_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # mean and std values of ImageNet channel
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

Two dataset classes are defined to support different input modalities.

LandCoverDataset handles both image only and image+text inputs. For each sample,
the image is loaded and transformed, and target values are extracted as a
multi-output regression vector. If a tokenizer and text column are provided,
captions are tokenized up to max_token_len tokens. The dataset returns an image
tensor, target vector, and optionally tokenized text inputs (input_ids,
attention_mask).

LandCoverRawTextDataset is used for RemoteCLIP experiments where tokenization
is handled internally by the text encoder. It returns the raw caption string
alongside the image tensor and target vector.

Dataset Classes

In [15]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=CONFIG["max_token_len"]):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [16]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (MaxVit) is used to extract image features, which
are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, a fully connected hidden layer
of size regressor_hidden_dim, ReLU activation, dropout at rate regressor_dropout,
and a final output layer. Both are configured via CONFIG.

This baseline allows comparison against multimodal models to measure the contribution
of textual information. The backbone is initialized with pretrained weights and
fine-tuned during training.

In [17]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=len(TARGET_COLS)):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information
through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (MaxVit) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT / RemoteCLIP) for caption representation
- A projection layer to map text features into the image feature space

MiniLM and DistilBERT are general-purpose transformer encoders that require
explicit tokenization. ImageTextGatedFrozenScaledModel handles these via
input_ids and attention_mask passed from the DataLoader. RemoteCLIP, by
contrast, is a CLIP-based model pretrained specifically on remote sensing imagery.
It handles tokenization internally, so it uses a separate model class
(MaxVitRemoteCLIPFusion) that accepts raw text strings directly.

The text encoder is kept frozen during training to reduce computational cost and
ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are
concatenated and passed through a learnable gate. The gate controls how much
textual information contributes to the final representation. A scaling factor
(text_scale) further adjusts the strength of text features, and is configured
via CONFIG.

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

The regression head hidden size and dropout rate are configured via
regressor_hidden_dim and regressor_dropout in CONFIG.

This design allows the model to adaptively use textual information depending on
its relevance.

In [18]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [19]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename=f"RemoteCLIP-{model_name}.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [20]:
class MaxVitRemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Function

This section defines a unified evaluation function for all model types.

A model_type parameter ("image", "text`, or "remoteclip") controls how
the batch is unpacked and passed to the model, allowing a single function to
handle all three experiment types consistently.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean
Squared Error (RMSE) is also computed for additional analysis. Class-wise MAE
is calculated to analyze performance across individual land cover categories.

These metrics provide both overall and class level performance insights.

In [21]:
@torch.no_grad()
def evaluate_model(model, loader, config, model_type):
    """
    Unified evaluation for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images  = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        if model_type == "image":
            preds = model(images)
        elif model_type == "text":
            input_ids      = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            preds = model(images, input_ids, attention_mask)
        elif model_type == "remoteclip":
            preds = model(images, batch["text"])

        loss = criterion(preds, targets)
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae       = mean_absolute_error(y_true, y_pred)
    rmse      = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training Function

This section defines a unified training loop for all model types.

A model_type parameter ("image", "text", or "remoteclip") controls how
the batch is unpacked and passed to the model.
The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with
weight decay is used. Gradient clipping is applied at each step. Early stopping
is triggered when validation MAE does not improve for early_stopping_patience
consecutive epochs, at which point the best model state is restored.

Training progress is logged to Weights & Biases (W&B) each epoch.

In [22]:
def train_model(model, train_loader, val_loader, config, model_type):
    """
    Unified training loop for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae    = float("inf")
    best_state      = None
    patience_counter = 0
    history         = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images  = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            if model_type == "image":
                preds = model(images)
            elif model_type == "text":
                input_ids      = batch["input_ids"].to(config["device"])
                attention_mask = batch["attention_mask"].to(config["device"])
                preds = model(images, input_ids, attention_mask)
            elif model_type == "remoteclip":
                preds = model(images, batch["text"])

            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_model(
            model, val_loader, config, model_type
        )

        history.append({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae     = val_mae
            best_state       = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_mae

## Experiment: Image Only Baseline

This function runs the image only experiment using the MaxVit backbone.

The full CONFIG is logged to W&B as the run configuration. The wandb_project
parameter defaults to CONFIG["wandb_project"] to avoid hardcoding. If
save_predictions is set to True, test set predictions and ground truth values
are saved as a CSV to CONFIG["predictions_dir"].

Metrics (MAE, RMSE, and class-wise MAE) are logged to Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [23]:
def run_maxvit_image_only(
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "maxvit_image_only",
        "fusion":       "none",
        "text_column":  "none",
        "text_model":   "none",
    }

    wandb.init(
        project=wandb_project,
        name="maxvit_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="image")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="image"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_image_only.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment": "maxvit_image_only",
        "text_model": "none",
        "test_mae":   test_mae,
        "test_rmse":  test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual
information using transformer-based text encoders (MiniLM, DistilBERT).

MiniLM and DistilBERT are lightweight transformer encoders that process tokenized
text. They require explicit tokenization before the forward pass, input IDs and
attention masks are prepared by the DataLoader via LandCoverDataset and passed
directly to the model. Therefore run_maxvit_transformer_text uses
model_type="text" and LandCoverDataset with a tokenizer.

RemoteCLIP, by contrast, handles its own tokenization internally inside the text
encoder's forward pass and expects raw caption strings. This is why it uses a
separate run_maxvit_remoteclip_text function with model_type="remoteclip"
and LandCoverRawTextDataset instead.

The text_col and text_model parameters specify the caption source and encoder
for each run. text_scale defaults to CONFIG["text_scale"] and wandb_project
defaults to CONFIG["wandb_project"] to avoid hardcoding. If save_predictions
is set to True, test set predictions are saved to CONFIG["predictions_dir"].

Results are logged to W&B, enabling comparison across different caption sources
and text encoders.

In [24]:
def run_maxvit_transformer_text(
    text_col,
    text_model,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "maxvit_transformer_text",
        "fusion":       "gated_frozen_scaled",
        "text_column":  text_col,
        "text_model":   text_model,
        "text_scale":   text_scale,
    }

    run_name = f"maxvit_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="text")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="text"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [25]:
def run_maxvit_remoteclip_text(
    text_col,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":  "maxvit_remoteclip_text",
        "fusion":      "gated_frozen_scaled",
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
    }

    run_name = f"maxvit_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = MaxVitRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="remoteclip")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="remoteclip"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [26]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [27]:
results = []

## Image Only Experiment

In [28]:
results.append(run_maxvit_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 11.5970 | Val MAE: 8.7794 | Val RMSE: 19.8260
Epoch 2/15 | Train Loss: 7.1607 | Val MAE: 4.7748 | Val RMSE: 11.2213
Epoch 3/15 | Train Loss: 4.8554 | Val MAE: 3.6786 | Val RMSE: 9.0202
Epoch 4/15 | Train Loss: 4.0314 | Val MAE: 3.0788 | Val RMSE: 8.1715
Epoch 5/15 | Train Loss: 3.7106 | Val MAE: 2.8026 | Val RMSE: 7.7370
Epoch 6/15 | Train Loss: 3.4604 | Val MAE: 2.7998 | Val RMSE: 7.4941
Epoch 7/15 | Train Loss: 3.1839 | Val MAE: 2.8799 | Val RMSE: 7.3402
Epoch 8/15 | Train Loss: 2.9293 | Val MAE: 2.8529 | Val RMSE: 7.6208
Epoch 9/15 | Train Loss: 2.7608 | Val MAE: 2.5178 | Val RMSE: 6.7081
Epoch 10/15 | Train Loss: 2.5713 | Val MAE: 2.4222 | Val RMSE: 6.3388
Epoch 11/15 | Train Loss: 2.4661 | Val MAE: 2.2860 | Val RMSE: 6.1710
Epoch 12/15 | Train Loss: 2.3447 | Val MAE: 2.3247 | Val RMSE: 6.1813
Epoch 13/15 | Train Loss: 2.2319 | Val MAE: 2.2099 | Val RMSE: 5.9516
Epoch 14/15 | Train Loss: 2.1679 | Val MAE: 2.2388 | Val RMSE: 6.1303
Epoch 15/15 | Train Loss: 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
epoch,15
test_loss,2.34998
test_mae,2.34998


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,maxvit_image_only,none,2.349979,6.163933,2.7464,0.956891,5.746852,4.121286,0.501601,1.99779,0.379033


## Multimodal Experiments
Multimodal experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

### MiniLM Experiments

In [29]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁
val_mae,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.94293
epoch,15


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Epoch 1/15 | Train Loss: 11.3991 | Val MAE: 8.3633 | Val RMSE: 18.9498
Epoch 2/15 | Train Loss: 6.8192 | Val MAE: 4.5651 | Val RMSE: 10.9347
Epoch 3/15 | Train Loss: 4.7161 | Val MAE: 3.3976 | Val RMSE: 8.6821
Epoch 4/15 | Train Loss: 4.0031 | Val MAE: 3.0738 | Val RMSE: 8.0614
Epoch 5/15 | Train Loss: 3.6894 | Val MAE: 2.9463 | Val RMSE: 8.3027
Epoch 6/15 | Train Loss: 3.4411 | Val MAE: 2.6580 | Val RMSE: 7.3301
Epoch 7/15 | Train Loss: 3.1697 | Val MAE: 2.8604 | Val RMSE: 7.3357
Epoch 8/15 | Train Loss: 2.9392 | Val MAE: 2.5528 | Val RMSE: 6.8611
Epoch 9/15 | Train Loss: 2.7783 | Val MAE: 2.6448 | Val RMSE: 6.7954
Epoch 10/15 | Train Loss: 2.6186 | Val MAE: 2.6277 | Val RMSE: 6.7409
Epoch 11/15 | Train Loss: 2.4963 | Val MAE: 2.3348 | Val RMSE: 6.3442
Epoch 12/15 | Train Loss: 2.3652 | Val MAE: 2.3234 | Val RMSE: 6.2711
Epoch 13/15 | Train Loss: 2.2778 | Val MAE: 2.4901 | Val RMSE: 6.3769
Epoch 14/15 | Train Loss: 2.1986 | Val MAE: 2.3025 | Val RMSE: 5.9005
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▂▂▂▁▂▁▁▁▁▁▁▁▁
val_mae,█▄▂▂▂▁▂▁▁▁▁▁▁▁▁
val_rmse,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,2.30253
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,sentence-transformers/all-MiniLM-L6-v2,2.005591,4.766330,2.522321,0.954387,4.822299,2.998545,0.478299,1.895850,0.367437,text_qwen3_4b_clean,0.7,1.942927,"[2.5223212, 0.9543867, 4.8222985, 2.9985452, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.254121,5.616295,2.731521,0.949566,5.340979,3.606718,0.509214,2.234483,0.406365,hybrid_gemma3_4b_clean,0.7,2.206304,"[2.7315207, 0.94956607, 5.340979, 3.6067183, 0..."
0,maxvit_image_only,none,2.349979,6.163933,2.746400,0.956891,5.746852,4.121286,0.501601,1.997790,0.379033,NaN,NaN,NaN,NaN
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.545931,6.466797,3.414648,0.955537,6.271712,4.135528,0.499667,2.073133,0.471294,vision_gemma3_4b_clean,0.7,2.302534,"[3.414648, 0.95553654, 6.2717123, 4.1355276, 0..."


### Remote CLIP Experiments

In [30]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_remoteclip_text(
            text_col=text_col,
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6156 | Val MAE: 8.6839 | Val RMSE: 19.6800
Epoch 2/15 | Train Loss: 7.1540 | Val MAE: 4.9466 | Val RMSE: 11.5309
Epoch 3/15 | Train Loss: 4.8802 | Val MAE: 3.4658 | Val RMSE: 8.6854
Epoch 4/15 | Train Loss: 4.0621 | Val MAE: 2.9237 | Val RMSE: 8.0025
Epoch 5/15 | Train Loss: 3.6202 | Val MAE: 2.9723 | Val RMSE: 7.9967
Epoch 6/15 | Train Loss: 3.3957 | Val MAE: 2.8980 | Val RMSE: 7.4719
Epoch 7/15 | Train Loss: 3.1689 | Val MAE: 2.5846 | Val RMSE: 6.8879
Epoch 8/15 | Train Loss: 2.8790 | Val MAE: 2.4528 | Val RMSE: 6.4978
Epoch 9/15 | Train Loss: 2.7114 | Val MAE: 2.5876 | Val RMSE: 6.5343
Epoch 10/15 | Train Loss: 2.5226 | Val MAE: 2.2032 | Val RMSE: 5.7081
Epoch 11/15 | Train Loss: 2.3922 | Val MAE: 2.1765 | Val RMSE: 5.6748
Epoch 12/15 | Train Loss: 2.2460 | Val MAE: 2.1213 | Val RMSE: 5.4348
Epoch 13/15 | Train Loss: 2.1373 | Val MAE: 2.0800 | Val RMSE: 5.3770
Epoch 14/15 | Train Loss: 2.0639 | Val MAE: 2.1013 | Val RMSE: 5.3828


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▁▂▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▁▂▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.01757
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6541 | Val MAE: 8.7696 | Val RMSE: 19.8778
Epoch 2/15 | Train Loss: 7.1760 | Val MAE: 4.7342 | Val RMSE: 11.3478
Epoch 3/15 | Train Loss: 4.8206 | Val MAE: 3.3457 | Val RMSE: 8.4638
Epoch 4/15 | Train Loss: 3.9396 | Val MAE: 3.4531 | Val RMSE: 9.0413
Epoch 5/15 | Train Loss: 3.4909 | Val MAE: 2.8728 | Val RMSE: 7.3655
Epoch 6/15 | Train Loss: 3.2530 | Val MAE: 2.7451 | Val RMSE: 6.7495
Epoch 7/15 | Train Loss: 3.0054 | Val MAE: 2.4516 | Val RMSE: 6.4065
Epoch 8/15 | Train Loss: 2.7394 | Val MAE: 2.2626 | Val RMSE: 5.7785
Epoch 9/15 | Train Loss: 2.5686 | Val MAE: 2.2037 | Val RMSE: 5.5978
Epoch 10/15 | Train Loss: 2.4173 | Val MAE: 2.2583 | Val RMSE: 5.7099
Epoch 11/15 | Train Loss: 2.2789 | Val MAE: 2.1088 | Val RMSE: 5.3444
Epoch 12/15 | Train Loss: 2.1693 | Val MAE: 1.9773 | Val RMSE: 4.9374
Epoch 13/15 | Train Loss: 2.0718 | Val MAE: 2.0830 | Val RMSE: 5.0014
Epoch 14/15 | Train Loss: 1.9878 | Val MAE: 1.8490 | Val RMSE: 4.5467


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▃▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▄▃▃▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▄▃▃▂▂▂▂▁▂▁▁▁▁▁
best_val_mae,1.84905
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 11.6299 | Val MAE: 8.6913 | Val RMSE: 19.7977
Epoch 2/15 | Train Loss: 7.1075 | Val MAE: 4.6970 | Val RMSE: 11.2006
Epoch 3/15 | Train Loss: 4.8498 | Val MAE: 3.3779 | Val RMSE: 8.5512
Epoch 4/15 | Train Loss: 4.0582 | Val MAE: 3.2382 | Val RMSE: 8.6566
Epoch 5/15 | Train Loss: 3.6700 | Val MAE: 2.8250 | Val RMSE: 7.7900
Epoch 6/15 | Train Loss: 3.4487 | Val MAE: 3.1083 | Val RMSE: 7.7870
Epoch 7/15 | Train Loss: 3.2559 | Val MAE: 2.7851 | Val RMSE: 7.4530
Epoch 8/15 | Train Loss: 3.0222 | Val MAE: 2.8512 | Val RMSE: 7.1488
Epoch 9/15 | Train Loss: 2.8496 | Val MAE: 2.5513 | Val RMSE: 6.8143
Epoch 10/15 | Train Loss: 2.6438 | Val MAE: 2.4645 | Val RMSE: 6.7775
Epoch 11/15 | Train Loss: 2.5157 | Val MAE: 2.4813 | Val RMSE: 6.5468
Epoch 12/15 | Train Loss: 2.3993 | Val MAE: 2.5139 | Val RMSE: 6.3878
Epoch 13/15 | Train Loss: 2.2899 | Val MAE: 2.3927 | Val RMSE: 6.2750
Epoch 14/15 | Train Loss: 2.1651 | Val MAE: 2.3287 | Val RMSE: 6.4305


best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▂▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,2.20013
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,sentence-transformers/all-MiniLM-L6-v2,2.005591,4.766330,2.522321,0.954387,4.822299,2.998545,0.478299,1.895850,0.367437,text_qwen3_4b_clean,0.7,1.942927,"[2.5223212, 0.9543867, 4.8222985, 2.9985452, 0..."
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.036035,4.847053,2.622733,0.944998,4.762298,3.087703,0.432208,2.002005,0.400300,text_qwen3_4b_clean,0.7,1.849049,"[2.622733, 0.94499767, 4.7622976, 3.0877035, 0..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.140530,5.473818,2.658745,0.952615,5.040355,3.376028,0.542306,2.019305,0.394355,hybrid_gemma3_4b_clean,0.7,2.017565,"[2.6587448, 0.95261484, 5.040355, 3.3760278, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.254121,5.616295,2.731521,0.949566,5.340979,3.606718,0.509214,2.234483,0.406365,hybrid_gemma3_4b_clean,0.7,2.206304,"[2.7315207, 0.94956607, 5.340979, 3.6067183, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.326058,6.288701,2.432545,0.952737,5.428941,4.261619,0.520094,2.216984,0.469486,vision_gemma3_4b_clean,0.7,2.200130,"[2.4325454, 0.9527373, 5.4289412, 4.2616186, 0..."
0,maxvit_image_only,none,2.349979,6.163933,2.746400,0.956891,5.746852,4.121286,0.501601,1.997790,0.379033,NaN,NaN,NaN,NaN
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.545931,6.466797,3.414648,0.955537,6.271712,4.135528,0.499667,2.073133,0.471294,vision_gemma3_4b_clean,0.7,2.302534,"[3.414648, 0.95553654, 6.2717123, 4.1355276, 0..."


### DistilBERT Experiments

In [31]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_maxvit_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.7387 | Val MAE: 9.0073 | Val RMSE: 20.3106
Epoch 2/15 | Train Loss: 7.3529 | Val MAE: 4.8106 | Val RMSE: 11.4959
Epoch 3/15 | Train Loss: 4.9632 | Val MAE: 4.0384 | Val RMSE: 9.7422
Epoch 4/15 | Train Loss: 4.2261 | Val MAE: 3.2099 | Val RMSE: 8.2089
Epoch 5/15 | Train Loss: 3.7677 | Val MAE: 2.8496 | Val RMSE: 7.5520
Epoch 6/15 | Train Loss: 3.4642 | Val MAE: 2.7649 | Val RMSE: 7.5388
Epoch 7/15 | Train Loss: 3.2318 | Val MAE: 2.6749 | Val RMSE: 7.0079
Epoch 8/15 | Train Loss: 3.0031 | Val MAE: 2.6988 | Val RMSE: 6.9194
Epoch 9/15 | Train Loss: 2.8220 | Val MAE: 2.7021 | Val RMSE: 7.3667
Epoch 10/15 | Train Loss: 2.6598 | Val MAE: 2.6531 | Val RMSE: 7.1558
Epoch 11/15 | Train Loss: 2.5497 | Val MAE: 2.5331 | Val RMSE: 6.3831
Epoch 12/15 | Train Loss: 2.4032 | Val MAE: 2.3405 | Val RMSE: 6.1264
Epoch 13/15 | Train Loss: 2.2733 | Val MAE: 2.3736 | Val RMSE: 6.4232
Epoch 14/15 | Train Loss: 2.1844 | Val MAE: 2.2416 | Val RMSE: 5.8322
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▂▂▁▁▁▁▁
best_val_mae,2.23515
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,distilbert-base-uncased,1.915706,4.578294,2.319682,0.950778,4.601264,3.030249,0.403727,1.738769,0.365472,text_qwen3_4b_clean,0.7,1.834685,"[2.3196824, 0.95077837, 4.6012635, 3.0302494, ..."
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,sentence-transformers/all-MiniLM-L6-v2,2.005591,4.766330,2.522321,0.954387,4.822299,2.998545,0.478299,1.895850,0.367437,text_qwen3_4b_clean,0.7,1.942927,"[2.5223212, 0.9543867, 4.8222985, 2.9985452, 0..."
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.036035,4.847053,2.622733,0.944998,4.762298,3.087703,0.432208,2.002005,0.400300,text_qwen3_4b_clean,0.7,1.849049,"[2.622733, 0.94499767, 4.7622976, 3.0877035, 0..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.140530,5.473818,2.658745,0.952615,5.040355,3.376028,0.542306,2.019305,0.394355,hybrid_gemma3_4b_clean,0.7,2.017565,"[2.6587448, 0.95261484, 5.040355, 3.3760278, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.254121,5.616295,2.731521,0.949566,5.340979,3.606718,0.509214,2.234483,0.406365,hybrid_gemma3_4b_clean,0.7,2.206304,"[2.7315207, 0.94956607, 5.340979, 3.6067183, 0..."
7,maxvit_hybrid_gemma3_4b_clean_distilbert-base-...,distilbert-base-uncased,2.281300,6.020039,2.518426,0.951406,5.475523,4.135687,0.417176,2.088873,0.382012,hybrid_gemma3_4b_clean,0.7,2.181172,"[2.5184262, 0.95140564, 5.4755225, 4.135687, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.326058,6.288701,2.432545,0.952737,5.428941,4.261619,0.520094,2.216984,0.469486,vision_gemma3_4b_clean,0.7,2.200130,"[2.4325454, 0.9527373, 5.4289412, 4.2616186, 0..."
0,maxvit_image_only,none,2.349979,6.163933,2.746400,0.956891,5.746852,4.121286,0.501601,1.997790,0.379033,NaN,NaN,NaN,NaN
9,maxvit_vision_gemma3_4b_clean_distilbert-base-...,distilbert-base-uncased,2.384807,6.284914,2.740020,0.955577,5.806966,4.297388,0.456191,2.023027,0.414477,vision_gemma3_4b_clean,0.7,2.235154,"[2.7400196, 0.95557743, 5.8069663, 4.297388, 0..."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.545931,6.466797,3.414648,0.955537,6.271712,4.135528,0.499667,2.073133,0.471294,vision_gemma3_4b_clean,0.7,2.302534,"[3.414648, 0.95553654, 6.2717123, 4.1355276, 0..."


### MaxVit Experiment Results Summary

In [32]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,distilbert-base-uncased,1.915706,4.578294,2.319682,0.950778,4.601264,3.030249,0.403727,1.738769,0.365472,text_qwen3_4b_clean,0.7,1.834685,"[2.3196824, 0.95077837, 4.6012635, 3.0302494, ..."
2,maxvit_text_qwen3_4b_clean_all-MiniLM-L6-v2_sc...,sentence-transformers/all-MiniLM-L6-v2,2.005591,4.766330,2.522321,0.954387,4.822299,2.998545,0.478299,1.895850,0.367437,text_qwen3_4b_clean,0.7,1.942927,"[2.5223212, 0.9543867, 4.8222985, 2.9985452, 0..."
5,maxvit_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.036035,4.847053,2.622733,0.944998,4.762298,3.087703,0.432208,2.002005,0.400300,text_qwen3_4b_clean,0.7,1.849049,"[2.622733, 0.94499767, 4.7622976, 3.0877035, 0..."
4,maxvit_remoteclip_hybrid_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.140530,5.473818,2.658745,0.952615,5.040355,3.376028,0.542306,2.019305,0.394355,hybrid_gemma3_4b_clean,0.7,2.017565,"[2.6587448, 0.95261484, 5.040355, 3.3760278, 0..."
1,maxvit_hybrid_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.254121,5.616295,2.731521,0.949566,5.340979,3.606718,0.509214,2.234483,0.406365,hybrid_gemma3_4b_clean,0.7,2.206304,"[2.7315207, 0.94956607, 5.340979, 3.6067183, 0..."
7,maxvit_hybrid_gemma3_4b_clean_distilbert-base-...,distilbert-base-uncased,2.281300,6.020039,2.518426,0.951406,5.475523,4.135687,0.417176,2.088873,0.382012,hybrid_gemma3_4b_clean,0.7,2.181172,"[2.5184262, 0.95140564, 5.4755225, 4.135687, 0..."
6,maxvit_remoteclip_vision_gemma3_4b_clean_scale...,RemoteCLIP ViT-B-32,2.326058,6.288701,2.432545,0.952737,5.428941,4.261619,0.520094,2.216984,0.469486,vision_gemma3_4b_clean,0.7,2.200130,"[2.4325454, 0.9527373, 5.4289412, 4.2616186, 0..."
0,maxvit_image_only,none,2.349979,6.163933,2.746400,0.956891,5.746852,4.121286,0.501601,1.997790,0.379033,NaN,NaN,NaN,NaN
9,maxvit_vision_gemma3_4b_clean_distilbert-base-...,distilbert-base-uncased,2.384807,6.284914,2.740020,0.955577,5.806966,4.297388,0.456191,2.023027,0.414477,vision_gemma3_4b_clean,0.7,2.235154,"[2.7400196, 0.95557743, 5.8069663, 4.297388, 0..."
3,maxvit_vision_gemma3_4b_clean_all-MiniLM-L6-v2...,sentence-transformers/all-MiniLM-L6-v2,2.545931,6.466797,3.414648,0.955537,6.271712,4.135528,0.499667,2.073133,0.471294,vision_gemma3_4b_clean,0.7,2.302534,"[3.414648, 0.95553654, 6.2717123, 4.1355276, 0..."


### MaxViT Results Summary

Experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

The image-only baseline achieves a test MAE of 2.36, slightly better than SwinV2, indicating stronger visual feature extraction.

Incorporating textual information leads to improvements, with the best performance obtained using text-only captions (MAE: ~1.94–1.99). However, the magnitude of improvement is smaller compared to SwinV2.

Hybrid captions provide limited gains over the baseline, and in some cases only marginal improvement. Vision-based captions again do not improve performance and remain close to the image only baseline.

Class-wise results show modest improvements, particularly for dominant classes such as Grass and Crop, while changes for other classes remain relatively small.

Overall, while textual information still improves performance, the gains are less pronounced compared to SwinV2, suggesting that the stronger MaxViT backbone already captures more informative visual features.

## Text Scale Experiments

The best caption × encoder combination for MaxVit based on
validation MAE at scale 0.7 is selected.

In [33]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for MaxVit (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for MaxVit (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 1.8347
  Test MAE    : 1.9157


### Text Scale Ablation

Using the best configuration selected above, we sweep text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5].

In [34]:
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_maxvit_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )
    else:
        result = run_maxvit_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.7304 | Val MAE: 9.0104 | Val RMSE: 20.2779
Epoch 2/15 | Train Loss: 7.2661 | Val MAE: 4.7330 | Val RMSE: 11.2289
Epoch 3/15 | Train Loss: 4.8822 | Val MAE: 3.8620 | Val RMSE: 9.2281
Epoch 4/15 | Train Loss: 4.0936 | Val MAE: 3.2402 | Val RMSE: 8.0132
Epoch 5/15 | Train Loss: 3.6672 | Val MAE: 2.7301 | Val RMSE: 7.2644
Epoch 6/15 | Train Loss: 3.2904 | Val MAE: 2.5607 | Val RMSE: 6.7335
Epoch 7/15 | Train Loss: 3.0237 | Val MAE: 2.4318 | Val RMSE: 6.3214
Epoch 8/15 | Train Loss: 2.8185 | Val MAE: 2.5964 | Val RMSE: 6.5697
Epoch 9/15 | Train Loss: 2.6473 | Val MAE: 2.3256 | Val RMSE: 5.7512
Epoch 10/15 | Train Loss: 2.4798 | Val MAE: 2.2081 | Val RMSE: 5.5038
Epoch 11/15 | Train Loss: 2.3572 | Val MAE: 2.0997 | Val RMSE: 5.2120
Epoch 12/15 | Train Loss: 2.2290 | Val MAE: 2.0720 | Val RMSE: 5.1973
Epoch 13/15 | Train Loss: 2.1281 | Val MAE: 2.0273 | Val RMSE: 5.0949
Epoch 14/15 | Train Loss: 2.0382 | Val MAE: 2.1192 | Val RMSE: 5.1507
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.97441
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 11.6991 | Val MAE: 8.9209 | Val RMSE: 20.0127
Epoch 2/15 | Train Loss: 7.1569 | Val MAE: 4.9171 | Val RMSE: 11.6830
Epoch 3/15 | Train Loss: 4.7701 | Val MAE: 4.0257 | Val RMSE: 9.3564
Epoch 4/15 | Train Loss: 3.9726 | Val MAE: 3.0157 | Val RMSE: 7.7224
Epoch 5/15 | Train Loss: 3.5293 | Val MAE: 2.7242 | Val RMSE: 7.0187
Epoch 6/15 | Train Loss: 3.2117 | Val MAE: 2.5147 | Val RMSE: 6.4673
Epoch 7/15 | Train Loss: 2.9916 | Val MAE: 2.4121 | Val RMSE: 6.1395
Epoch 8/15 | Train Loss: 2.7606 | Val MAE: 2.3987 | Val RMSE: 5.7574
Epoch 9/15 | Train Loss: 2.5473 | Val MAE: 2.2967 | Val RMSE: 5.4502
Epoch 10/15 | Train Loss: 2.3993 | Val MAE: 2.1354 | Val RMSE: 5.2439
Epoch 11/15 | Train Loss: 2.2952 | Val MAE: 2.0553 | Val RMSE: 4.9279
Epoch 12/15 | Train Loss: 2.1892 | Val MAE: 1.9985 | Val RMSE: 4.8743
Epoch 13/15 | Train Loss: 2.0870 | Val MAE: 2.0191 | Val RMSE: 4.8592
Epoch 14/15 | Train Loss: 1.9960 | Val MAE: 1.8560 | Val RMSE: 4.5957
Epoch 15/15 | Train Loss: 

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,1.80904
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,1.974412,2.121737,5.239959,"[2.81151, 0.95118636, 5.0329, 3.3298092, 0.421...",2.811510,0.951186,5.032900,3.329809,0.421239,1.963090,0.342422
1,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,1.847067,1.956103,4.756630,"[2.4085972, 0.9512677, 4.539476, 3.0147789, 0....",2.408597,0.951268,4.539476,3.014779,0.483079,1.909484,0.386040
2,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.922825,1.955957,4.684213,"[2.4412415, 0.95128554, 4.71124, 2.9569721, 0....",2.441242,0.951286,4.711240,2.956972,0.434823,1.811257,0.384878
3,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,1.902689,2.031114,4.936973,"[2.547663, 0.9486902, 4.654006, 3.465789, 0.41...",2.547663,0.948690,4.654006,3.465789,0.411672,1.792572,0.397406
4,maxvit_text_qwen3_4b_clean_distilbert-base-unc...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,1.809037,1.907849,4.712403,"[2.3883808, 0.9517061, 4.4448905, 2.915844, 0....",2.388381,0.951706,4.444890,2.915844,0.473258,1.804844,0.376021


### Text Scale Ablation — Overall MAE Summary

In [35]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.3500

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,1.974412,2.121737,5.239959,0.2282,9.71
1,0.5,1.847067,1.956103,4.756630,0.3939,16.76
2,0.7,1.922825,1.955957,4.684213,0.3940,16.77
3,1.0,1.902689,2.031114,4.936973,0.3189,13.57
4,1.5,1.809037,1.907849,4.712403,0.4421,18.81


### Text Scale Ablation — Class-wise MAE

In [36]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.811510 0.951186 5.032900 3.329809  0.421239 1.963090 0.342422
        0.5 2.408597 0.951268 4.539476 3.014779  0.483079 1.909484 0.386040
        0.7 2.441242 0.951286 4.711240 2.956972  0.434823 1.811257 0.384878
        1.0 2.547663 0.948690 4.654006 3.465789  0.411672 1.792572 0.397406
        1.5 2.388381 0.951706 4.444890 2.915844  0.473258 1.804844 0.376021

Class-wise improvement vs image-only baseline (positive = better)
 text_scale    Tree  Shrub  Grass   Crop  Built-up  Barren   Water
        0.3 -0.0651 0.0057 0.7140 0.7915    0.0804  0.0347  0.0366
        0.5  0.3378 0.0056 1.2074 1.1065    0.0185  0.0883 -0.0070
        0.7  0.3052 0.0056 1.0356 1.1643    0.0668  0.1865 -0.0058
        1.0  0.1987 0.0082 1.0928 0.6555    0.0899  0.2052 -0.0184
        1.5  0.3580 0.0052 1.3020 1.2054    0.0283  0.1929  0.0030


## Summary of All MaxVit Experiments

In [37]:
scale_experiment_ids = {id(r) for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()
all_results_df["_source"] = (
    ["results"] * len(results) + ["scale_sweep"] * len(scale_results)
)

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"   if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"      if r["_source"] == "scale_sweep"
    else  "multimodal_0.7"),
    axis=1
)

all_results_df = all_results_df.drop(columns=["_source"])

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,none,NaN,NaN,2.349979,6.163933
1,multimodal_0.7,text_qwen3_4b_clean,distilbert-base-uncased,0.7,1.834685,1.915706,4.578294
2,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,1.942927,2.005591,4.766330
3,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,1.849049,2.036035,4.847053
4,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.017565,2.140530,5.473818
5,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.206304,2.254121,5.616295
6,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.181172,2.281300,6.020039
7,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.200130,2.326058,6.288701
8,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.235154,2.384807,6.284914
9,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.302534,2.545931,6.466797


In [38]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 1.5
  Val MAE         : 1.8090
  Test MAE        : 1.9078
  Test RMSE       : 4.7124


In [39]:
BACKBONE_NAME = CONFIG["model_name"]
save_path = os.path.join(CONFIG["predictions_dir"], f"all_results_{BACKBONE_NAME}.csv")
all_results_df.to_csv(save_path, index=False)
print("Saved results to Drive")

Saved results to Drive
